# Two-File Pipeline (with Explicit Psych & Demographic Renaming)

**Inputs**: `Aircontrol_with_SurveyAnswers_HR.csv`, `onboarding_survey.csv`

**Outputs**:
- `final_demographic_data.csv`
- `final_psychological_data.csv`
- `cold_start_data.csv`
- `final_demographic_data_full.csv`
- `final_psychological_data_full.csv`
- `final_psychological_data_CLEAN.csv`
- `final_main_dataset.csv`
- `final_main_dataset_imputed.csv`
- `final_main_dataset_no_missing.csv`
- `final_main_dataset_imputed_ENCODED.csv`
- `final_main_dataset_imputed_ALLENC.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
from anchor import data_dir

pd.set_option("display.max_columns", 200)

In [ ]:

DATA_PATH = data_dir()

DOWN_PATH = os.path.join(DATA_PATH,"all_aircontrol_with_surveys_hr_with_new_bpm_new.csv") #all_aircontrol_with_surveys_hr_FIXED
# ONBOARD_PATH =  os.path.join(DATA_PATH,"onboarding_survey.csv")
DOWN_ONBOARD =  os.path.join(DATA_PATH,"final_onboarding_survey.csv")
ONBOARD_PATH_NEW =  os.path.join(DATA_PATH,"scores.csv")

DORN_QNS =  os.path.join(DATA_PATH,"dorn","DORN_qns_b5p_calc_fixed.csv")
DORN_QNS_SWLS = os.path.join(DATA_PATH,"dorn","DORN_qns_hsps_swls_phy.csv")

OUT_DEMO =  os.path.join(DATA_PATH,"final_demographic_data.csv")
OUT_DEMO_FULL =  os.path.join(DATA_PATH,"final_demographic_data_full.csv")
OUT_PSY =  os.path.join(DATA_PATH,"final_psychological_data.csv")
OUT_PSY_FULL =  os.path.join(DATA_PATH,"final_psychological_data_full.csv")


OUT_COLD =  os.path.join(DATA_PATH,"cold_start_data.csv")
OUT_COLD_FULL = os.path.join(DATA_PATH,"cold_start_data_full.csv")
OUT_COLD_FULL_SURVEY = os.path.join(DATA_PATH,"cold_start_data_full_survey.csv")
OUT_COLD_FULL_BIG5 = os.path.join(DATA_PATH,"cold_start_data_full_big5.csv")


OUT_MAIN =  os.path.join(DATA_PATH,"final_main_dataset_new_bpm.csv")
OUT_NO_MISSING =  os.path.join(DATA_PATH,"final_main_dataset_no_missing_new_bpm.csv")
OUT_NO_MISSING_ALLENC =  os.path.join(DATA_PATH,"final_main_dataset_no_missing_ALLENC_new_bpm.csv")

OUT_COLD_FILTERED =  os.path.join(DATA_PATH,"thermal_pref_filtered_cold.csv")
OUT_NO_MISSING_ALLENC_FILTERED =  os.path.join(DATA_PATH,"thermal_pref_filtered_sensors.csv")

OUT_DORN_ALL =  os.path.join(DATA_PATH,"dorn","DORN_qns_b5p_calc_sex_swls_fixed.csv")
# OUT_DORN_SURVEY =  os.path.join(DATA_PATH,"dorn","DORN_qns_b5p_calc_swls_fixed.csv")


WITH_TIME = os.path.join(DATA_PATH,"final_main_dataset_no_missing_ALLENC_with_time_new_bpm.csv")
WITH_HR_AGE = os.path.join(DATA_PATH,"final_main_dataset_no_missing_ALLENC_with_time_and_hr_age.csv")

OUT_DIR = os.path.join(DATA_PATH,"sensor_subsets")


In [ ]:
sensors_df = pd.read_csv(DOWN_PATH, usecols=["user_id"])
onb_df = pd.read_csv(ONBOARD_PATH_NEW)

# Keep only rows where user_id exists in the sensors
matching_ids = set(sensors_df["user_id"].dropna().unique())
onb_filtered = onb_df[onb_df["id"].isin(matching_ids)].copy()

onb_filtered.to_csv(DOWN_ONBOARD, index=False)

## Load & Rename Onboarding (explicit map)

In [ ]:
onb = pd.read_csv(DOWN_ONBOARD)
onb.columns = [c.strip() for c in onb.columns]

rename_map = {
    'id': 'user_id',
    'Geschlecht:': 'sex',
    'Sind Sie aktuell schwanger?': 'pregnant',
    'Alter:': 'age',
    'Höchster Bildungsabschluss:': 'education_level',
    'Haben Sie mehrere Jahre (5+) außerhalb von Österreich gelebt?': 'lived_abroad',
    'In welchem Land haben Sie gelebt und seit wann sind Sie (wieder) in Österreich?': 'country_abroad_and_return_date',
    'Wohnort (PLZ, Ort):': 'residence_location',
    'Art der Wohnimmobilie:': 'housing_type',
    'Baualter der Immobilie': 'building_age',
    'Bitte geben Sie die Bauperiode der Immobilie an, sofern Sie diese kennen:': 'building_period',
    'Klimatisierungsmöglichkeit:': 'ac_available',
    'Lüftung und Lüftungsverhalten:': 'ventilation_behavior',
    'Heizungssystem:': 'heating_system',
    'Wurde die Immobilie in den letzten 20 Jahren saniert?': 'renovated_last_20_years',
    'Falls ja, wann?': 'renovation_year',
    'Umfang der Sanierung': 'renovation_extent',
    'Wie würden Sie das durchschnittliche Raumklima in Ihrem Zuhause einschätzen?\nUnter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.': 'home_comfort_rating',
    'Berufliche Tätigkeit (vorwiegend):': 'main_occupation',
    'Wie viele Stunden arbeiten Sie durchschnittlich pro Woche?': 'work_hours_per_week',
    'Arbeitsort (PLZ, Ort):Falls Sie mehrere Arbeitsorte haben, nennen Sie bitte den, an dem Sie die meiste Zeit verbringen.': 'main_work_location',
    'Welche Arbeitsorte nutzen Sie regelmäßig im Rahmen Ihrer Arbeit? (Mehrauswahl möglich)': 'regular_work_locations',
    'An welchen Wochentagen arbeiten Sie normalerweise?': 'usual_workdays',
    'An welchen Wochentagen arbeiten Sie normalerweise im Büro?': 'office_days',
    'An welchen Wochentagen arbeiten Sie normalerweise\xa0im Home-Office?': 'home_office_days',
    'An welchen Wochentagen arbeiten Sie normalerweise\xa0remote (zB in Cafés, in der Bibliothek)?': 'remote_work_days',
    'Geben Sie bitte die Räumlichkeiten in Ihrem Büro an:': 'office_spaces',
    'Klimatisierungsmöglichkeit im Büro:': 'ac_office',
    'Lüftung und Lüftungsverhalten im Büro:': 'ventilation_office',
    'Heizungssystem im Büro:': 'heating_office',
    'Wie würden\xa0 Sie das durchschnittliche Raumklima in Ihrem Büro einschätzen?\nUnter Raumklima verstehen wir Faktoren wie Temperatur, Luftfeuchtigkeit, Lichtverhältnisse, Lärmpegel, Luftfrische usw.': 'office_comfort_rating',

    'Körpergröße (in cm):': 'height_cm',
    'Gewicht (in kg):': 'weight_kg',
    'Wie schätzen Sie Ihr Fitnesslevel ein?': 'fitness_level',
    'Hitze_Tolleranz': 'heat_tolerance',
    'Kälte_Tolleranz': 'cold_tolerance',
    'SWLS': 'swls_score',

    # MAIA
    'MAIA_Bemerken': 'maia_noticing',
    'MAIA_Nicht-Ablenken': 'maia_not_distracting',
    'MAIA_ Sich-Keine-Sorgen-Machen': 'maia_not_worrying',
    'MAIA_Aufmerksamkeitsregulation': 'maia_attention_regulation',
    'MAIA_Emotionales Gewahrsein': 'maia_emotional_awareness',
    'MAIA_Selbstregulation': 'maia_self_regulation',
    'MAIA_Auf-den-Leib-Hören': 'maia_body_listening',
    'MAIA_Vertrauen': 'maia_trusting',

    # BFI2S
    'BFI2S_Extraversion': 'bfi2s_extraversion',
    'BFI2S_Geselligkeit': 'bfi2s_sociability',
    'BFI2S_Durchsetzungsfähigkeit': 'bfi2s_assertiveness',
    'BFI2S_Aktivität': 'bfi2s_activity',
    'BFI2S_Verträglichkeit': 'bfi2s_agreeableness',
    'BFI2S_Mitgefühl': 'bfi2s_compassion',
    'BFI2S_Höflichkeit': 'bfi2s_politeness',
    'BFI2S_Vertrauen': 'bfi2s_trust',
    'BFI2S_Gewissenhaftigkeit': 'bfi2s_conscientiousness',
    'BFI2S_Ordnungsliebe': 'bfi2s_orderliness',
    'BFI2S_Fleiß': 'bfi2s_industriousness',
    'BFI2S_Verlässlichkeit': 'bfi2s_reliability',
    'BFI2S_Negative Emotionalität': 'bfi2s_negative_emotionality',
    'BFI2S_Emotionalität': 'bfi2s_emotionality',
    'BFI2S_Ängstlichkeit': 'bfi2s_anxiety',
    'BFI2S_Niedergeschlagenheit': 'bfi2s_depression',
    'BFI2S_Unbeständigkeit der Gefühle': 'bfi2s_emotional_instability',
    'BFI2S_Offenheit': 'bfi2s_openness',
    'BFI2S_Ästhetisches Empfinden': 'bfi2s_aesthetic_sensitivity',
    'BFI2S_Intellektuelle Neugierde': 'bfi2s_intellectual_curiosity',
    'BFI2S_ Kreativer Einfallsreichtum': 'bfi2s_creative_imagination',

    # BFI2XS
    'BFI2XS_Extraversion': 'bfi2xs_extraversion',
    'BFI2XS_Verträglichkeit': 'bfi2xs_agreeableness',
    'BFI2XS_Gewissenhaftigkeit': 'bfi2xs_conscientiousness',
    'BFI2XS_Negative Emotionalität': 'bfi2xs_negative_emotionality',
    'BFI2XS_Emotionalität': 'bfi2xs_emotionality',
    'BFI2XS_Offenheit': 'bfi2xs_openness',

    # BSI
    'BSI_Somatization': 'bsi_somatization',
    'BSI_Obsession-Compulsion': 'bsi_obsession_compulsion',
    'BSI_Interpersonal Sensitivity': 'bsi_interpersonal_sensitivity',
    'BSI_Depression': 'bsi_depression',
    'BSI_Anxiety': 'bsi_anxiety',
    'BSI_Hostility': 'bsi_hostility',
    'BSI_Phobic Anxiety': 'bsi_phobic_anxiety',
    'BSI_Paranoid Ideation': 'bsi_paranoid_ideation',
    'BSI_Psychoticism': 'bsi_psychoticism',
    'BSI_GSI': 'bsi_gsi',
    'BSI_PST': 'bsi_pst',
    'BSI_PDSI_temp': 'bsi_pdsi_temp',
    'BSI_PDSI': 'bsi_pdsi',
}


onb = onb.rename(columns=rename_map)

# Normalize sex to English where needed (already English after rename; keep a safety map)
sex_map_de = {"Männlich": "Male", "Weiblich": "Female", "Divers": "Other"}
if "sex" in onb.columns:
    onb["sex"] = onb["sex"].map(sex_map_de).fillna(onb["sex"])

# Ensure numeric types for height/weight/age if present
for c in ["height_cm", "weight_kg", "age"]:
    if c in onb.columns:
        onb[c] = pd.to_numeric(onb[c], errors="coerce")

onb.head(2)


## Build Demographics

In [ ]:
# Build demographics table (one row per user)
demo_keep_full = [
    "user_id", "sex", "fitness_level", "height_cm", "weight_kg",
    "age", "pregnant", "education_level", "lived_abroad",
    "country_abroad_and_return_date", "residence_location",
    "housing_type", "building_age", "building_period", "ac_available",
    "ventilation_behavior", "heating_system", "renovated_last_20_years",
    "renovation_year", "renovation_extent", "home_comfort_rating",
    "main_occupation", "work_hours_per_week", "main_work_location",
    "regular_work_locations", "usual_workdays", "office_days",
    "home_office_days", "remote_work_days", "office_spaces",
    "ac_office", "ventilation_office", "heating_office", "office_comfort_rating",
    "heat_tolerance", "cold_tolerance"
]
demo_cols = [c for c in demo_keep_full if c in onb.columns]
demo_full = onb[demo_cols].drop_duplicates(subset=["user_id"])
demo_full.to_csv(OUT_DEMO_FULL, index=False)
demo_full.head(3)


In [ ]:
# Build demographics table (one row per user)
demo_keep = [
    "user_id", "sex", "age", "height_cm", "weight_kg", 
    "fitness_level", "heat_tolerance", "cold_tolerance"
]
demo_cols = [c for c in demo_keep if c in onb.columns]
demo = onb[demo_cols].drop_duplicates(subset=["user_id"])
demo.to_csv(OUT_DEMO, index=False)
demo.head(3)

## Build Psychological Datasets (scored columns only)

In [ ]:

# all psychological data
psych_targets_full = [
    "swls_score",
    # MAIA
    "maia_noticing","maia_not_distracting","maia_not_worrying","maia_attention_regulation",
    "maia_emotional_awareness","maia_self_regulation","maia_body_listening","maia_trusting",
    # BFI2S
    "bfi2s_extraversion","bfi2s_sociability","bfi2s_assertiveness","bfi2s_activity",
    "bfi2s_agreeableness","bfi2s_compassion","bfi2s_politeness","bfi2s_trust",
    "bfi2s_conscientiousness","bfi2s_orderliness","bfi2s_industriousness","bfi2s_reliability",
    "bfi2s_negative_emotionality","bfi2s_emotionality","bfi2s_anxiety","bfi2s_depression","bfi2s_emotional_instability",
    "bfi2s_openness","bfi2s_aesthetic_sensitivity","bfi2s_intellectual_curiosity","bfi2s_creative_imagination",
    # BFI2XS (if present)
    "bfi2xs_extraversion","bfi2xs_agreeableness","bfi2xs_conscientiousness","bfi2xs_negative_emotionality","bfi2xs_emotionality","bfi2xs_openness",
    # BSI
    "bsi_somatization","bsi_obsession_compulsion","bsi_interpersonal_sensitivity","bsi_depression","bsi_anxiety",
    "bsi_hostility","bsi_phobic_anxiety","bsi_paranoid_ideation","bsi_psychoticism","bsi_gsi","bsi_pst","bsi_pdsi_temp","bsi_pdsi"
]

present_psy_cols = ["user_id"] + [c for c in psych_targets_full if c in onb.columns]
psy_full = onb[present_psy_cols].copy()

# Optionally coerce numeric for psych scores
for c in present_psy_cols:
    if c != "user_id":
        psy_full[c] = pd.to_numeric(psy_full[c], errors="coerce")

psy_full.to_csv(OUT_PSY_FULL, index=False)

psy_full.shape, present_psy_cols[:10]


In [ ]:
# all psychological data
psych_targets_full = [
    "swls_score",
    # MAIA
    "maia_noticing","maia_not_distracting","maia_not_worrying","maia_attention_regulation",
    "maia_emotional_awareness","maia_self_regulation","maia_body_listening","maia_trusting",
    # BFI2S
    "bfi2s_extraversion","bfi2s_sociability","bfi2s_assertiveness","bfi2s_activity",
    "bfi2s_agreeableness","bfi2s_compassion","bfi2s_politeness","bfi2s_trust",
    "bfi2s_conscientiousness","bfi2s_orderliness","bfi2s_industriousness","bfi2s_reliability",
    "bfi2s_negative_emotionality","bfi2s_emotionality","bfi2s_anxiety","bfi2s_depression","bfi2s_emotional_instability",
    "bfi2s_openness","bfi2s_aesthetic_sensitivity","bfi2s_intellectual_curiosity","bfi2s_creative_imagination",
    # BFI2XS (if present)
    "bfi2xs_extraversion","bfi2xs_agreeableness","bfi2xs_conscientiousness","bfi2xs_negative_emotionality","bfi2xs_emotionality","bfi2xs_openness",
    # BSI
    "bsi_somatization","bsi_obsession_compulsion","bsi_interpersonal_sensitivity","bsi_depression","bsi_anxiety",
    "bsi_hostility","bsi_phobic_anxiety","bsi_paranoid_ideation","bsi_psychoticism","bsi_gsi","bsi_pst","bsi_pdsi_temp","bsi_pdsi"
]

present_psy_cols = ["user_id"] + [c for c in psych_targets_full if c in onb.columns]
psy_full = onb[present_psy_cols].copy()

# Optionally coerce numeric for psych scores
for c in present_psy_cols:
    if c != "user_id":
        psy_full[c] = pd.to_numeric(psy_full[c], errors="coerce")

In [ ]:
# a subset of psychological data
psych_targets = [
    "swls_score",
    # MAIA
    "maia_noticing","maia_not_distracting","maia_not_worrying","maia_attention_regulation",
    "maia_emotional_awareness","maia_self_regulation","maia_body_listening","maia_trusting",
    # BFI2S
    "bfi2s_sociability","bfi2s_assertiveness","bfi2s_activity", "bfi2s_compassion",
    "bfi2s_politeness","bfi2s_trust", "bfi2s_orderliness", "bfi2s_industriousness",
    "bfi2s_reliability", "bfi2s_anxiety","bfi2s_depression", "bfi2s_emotional_instability","bfi2s_emotionality",
    "bfi2s_aesthetic_sensitivity","bfi2s_intellectual_curiosity","bfi2s_creative_imagination",
]

present_psy_cols = ["user_id"] + [c for c in psych_targets if c in onb.columns]
psy = onb[present_psy_cols].copy()

# Optionally coerce numeric for psych scores
for c in present_psy_cols:
    if c != "user_id":
        psy[c] = pd.to_numeric(psy[c], errors="coerce")

psy.to_csv(OUT_PSY, index=False)

psy.shape, present_psy_cols[:10]


# reduce psychological dataframes so that one can do survey and big5

In [ ]:
# fill_psy = pd.read_csv(OUT_PSY_FULL)
# fill_psy.head()

# fill_psy.loc[:,["swls_score","bfi2s_extraversion","bfi2s_agreeableness","bfi2s_conscientiousness","bfi2s_emotionality","bfi2s_openness"]].to_csv(OUT_PSY_SURVEY, index=False)
# fill_psy.loc[:,["bfi2s_extraversion","bfi2s_agreeableness","bfi2s_conscientiousness","bfi2s_emotionality","bfi2s_openness"]].to_csv(OUT_PSY_BIG5, index=False)

## Load Sensors, Merge, Save Main

In [ ]:

# Load sensors
raw = pd.read_csv(DOWN_PATH)
raw.columns = [c.strip() for c in raw.columns]

# Rename sensors/survey -> canonical
rename_sensors = {}
if "AirTemp" in raw.columns:    rename_sensors["AirTemp"] = "t_ubi"
if "Humidity" in raw.columns:   rename_sensors["Humidity"] = "rh_ubi"
if "bpm" in raw.columns:        rename_sensors["bpm"] = "hr_ubi"
if "q_activity" in raw.columns: rename_sensors["q_activity"] = "activity"
if "CO2" in raw.columns: rename_sensors["CO2"] = "CO2"
if "survey_time" in raw.columns: rename_sensors["survey_time"] = "timestamp"
if "bpm_time_diff_log" in raw.columns: rename_sensors["bpm_time_diff_log"] = "bpm_time_diff_log"

raw = raw.rename(columns=rename_sensors)

# Build thermal_cozie
if "thermal_preference_num" in raw.columns:
    raw["thermal_cozie"] = raw["thermal_preference_num"].map({-1: 9, 0: 10, 1: 11})
elif "q_preference_temperature" in raw.columns:
    raw["thermal_cozie"] = raw["q_preference_temperature"].map({
        "Prefer Warmer": 9, "Prefer warmer": 9, "Warmer": 9, "Wärmer":9,
        "No Change": 10, "No change": 10, "Keine Veränderung": 10,
        "Prefer Cooler": 11, "Prefer cooler": 11, "Cooler": 11, "Kühler": 11
    })
else:
    raise ValueError("Need 'thermal_preference_num' or 'thermal_preference' in file.")

# Merge with demographics
env = raw.merge(demo, on="user_id", how="left", suffixes=("", "_demo"))

# Final column order
keep_cols = ["user_id","timestamp", "t_ubi","rh_ubi","hr_ubi","CO2","thermal_cozie",
             "activity","sex","fitness_level","height_cm","weight_kg", "bpm_time_diff_log"]
for col in keep_cols:
    if col not in env.columns:
        env[col] = pd.NA
env = env[keep_cols].copy()

# Save
env.to_csv(OUT_MAIN, index=False)
env.shape, env.head(2)


## Drop rows with missing values

In [ ]:
no_missing = env.copy()
no_missing = no_missing.dropna(axis=0)
no_missing.to_csv(OUT_NO_MISSING, index=False)

## Encoded Variants (dropped version)

In [ ]:
no_missing_enc = no_missing.copy()
activity_map = {
    "Erholungsphase (Lesen, Fernsehen, etc.)": 0,
    "Schreibtischarbeit (Bürotätigkeiten, etc.)": 1,
    "Sonstige Aktivität": 2,
}
no_missing_enc["activity"] = no_missing_enc["activity"].map(activity_map).astype("Int64")

sex_map = {"Male": 0, "Female": 1, "Other": 2}
no_missing_enc["sex"] = no_missing_enc["sex"].map(sex_map).astype("Int64")

fitness_map = {
    "Gering (wenig bis keine körperliche Aktivität)": 0,
    "Durchschnittlich (regelmäßige körperliche Aktivität)": 1,
    "Hoch (intensive körperliche Aktivität mehrmals pro Woche)": 2,
    "Sehr hoch (täglich intensive körperliche Aktivität)": 3,
}
no_missing_enc["fitness_level"] = no_missing_enc["fitness_level"].map(fitness_map).astype("Int64")

no_missing_enc.to_csv(WITH_HR_AGE, index=False) #with timestamp and hr/age
no_missing_enc.drop(columns=["bpm_time_diff_log"], inplace=True)
no_missing_enc.to_csv(WITH_TIME, index=False) #with timestamp
no_missing_enc.drop(columns=["timestamp"], inplace=True)
no_missing_enc.to_csv(OUT_NO_MISSING_ALLENC, index=False) #without timestamp

## Create cold start dataset

In [ ]:
# creating a file with feature for cold start combined
join_key = next((k for k in ["user_id", "participant_id", "occupant_id", "subject_id", "id"]
                 if k in psy.columns and k in demo.columns), None)
if join_key is None:
    raise KeyError("No common join key found.")

# Select required demo columns
demo_cols = ["sex", "age", "height_cm", "weight_kg", "fitness_level", "heat_tolerance", "cold_tolerance"]
demo_subset = demo[[join_key] + demo_cols].copy()

# Map categorical columns
demo_subset["sex"] = demo_subset["sex"].map(sex_map).astype("Int64")
demo_subset["fitness_level"] = demo_subset["fitness_level"].map(fitness_map).astype("Int64")

# Merge into psych file
psych_with_demo = pd.merge(psy, demo_subset, on=join_key, how="left")

# Save
psych_with_demo.to_csv(OUT_COLD, index=False)

In [ ]:
# creating a file with feature for cold start combined
join_key = next((k for k in ["user_id", "participant_id", "occupant_id", "subject_id", "id"]
                 if k in psy_full.columns and k in demo.columns), None)
if join_key is None:
    raise KeyError("No common join key found.")

# Select required demo columns
demo_cols = ["sex", "age", "height_cm", "weight_kg", "fitness_level", "heat_tolerance", "cold_tolerance"]
demo_subset = demo[[join_key] + demo_cols].copy()

# Map categorical columns
demo_subset["sex"] = demo_subset["sex"].map(sex_map).astype("Int64")
demo_subset["fitness_level"] = demo_subset["fitness_level"].map(fitness_map).astype("Int64")

# Merge into psych file
psych_full_with_demo = pd.merge(psy_full, demo_subset, on=join_key, how="left")

# Save
psych_full_with_demo.to_csv(OUT_COLD_FULL, index=False)

In [ ]:
# Save
psych_full_with_demo.loc[:,["user_id","swls_score","bfi2s_extraversion","bfi2s_agreeableness","bfi2s_conscientiousness","bfi2s_emotionality","bfi2s_openness"]].to_csv(OUT_COLD_FULL_SURVEY, index=False)
psych_full_with_demo.loc[:,["user_id","bfi2s_extraversion","bfi2s_agreeableness","bfi2s_conscientiousness","bfi2s_emotionality","bfi2s_openness"]].to_csv(OUT_COLD_FULL_BIG5, index=False)


## Class Imbalance handling

### Thermal preference answers

In [ ]:
import matplotlib.pyplot as plt

df = pd.read_csv(DOWN_PATH)
#pref_map = {9: "Prefer Warmer", 10: "No Change", 11: "Prefer Cooler"}
#df["q_preference_temperature"] = df["q_preference_temperature"].map(pref_map)

counts = df.groupby(["user_id", "q_preference_temperature"]).size().reset_index(name="count")
pivot = counts.pivot(index="user_id", columns="q_preference_temperature", values="count").fillna(0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

ax = pivot_pct.plot(
    kind="barh", stacked=True, figsize=(8,12),
    color={"Kühler":"#1f77b4", "Keine Veränderung":"#ff7f0e", "Wärmer":"#2ca02c"}
)
ax.set_xlabel("Percentage of answers (%)")
ax.set_ylabel("User")
ax.set_title("Thermal Preference Distribution per User (%)")
ax.legend(title="Preference", bbox_to_anchor=(1.05,1), loc="lower right")
plt.tight_layout()
plt.show()



### Count of answers per user

In [ ]:
# --- Total number of answers per user (horizontal) ---
totals = pivot.sum(axis=1)  # keep original user order

ax = totals.plot(kind="barh", figsize=(8,10), color="gray")
ax.set_xlabel("Number of answers")
ax.set_ylabel("User")
ax.set_title("Total answers per user")

# Find max and two mins
max_user = totals.idxmax()
max_val = totals[max_user]
min_users = totals.nsmallest(2)

# Annotate max (horizontal position = value, vertical = bar index)
ax.patches[totals.index.get_loc(max_user)].set_color("green")
ax.text(max_val + 1, totals.index.get_loc(max_user), f"{max_val}",
        va="center", ha="left", fontsize=10, color="green")

# Annotate two mins
for u, v in min_users.items():
    ax.patches[totals.index.get_loc(u)].set_color("red")
    ax.text(v + 1, totals.index.get_loc(u), f"{v}",
            va="center", ha="left", fontsize=10, color="red")

plt.tight_layout()
plt.show()


In [ ]:
# === Users with missing categories & users with <5% categories ===

# Make sure the three categories exist as columns (fill missing with 0)
cats = ["Kühler", "Keine Veränderung", "Wärmer"]
pivot_safe = pivot.reindex(columns=cats, fill_value=0)

# 1) Users NOT having data in all three categories (i.e., at least one is zero)
mask_missing_any = (pivot_safe[cats] == 0).any(axis=1)
users_missing_categories = pivot_safe.index[mask_missing_any].tolist()

# 2) Users having all three categories BUT one or more categories < 5% of that user's answers
have_all_three = (pivot_safe[cats] > 0).all(axis=1)
pct_safe = pivot_pct.reindex(columns=cats, fill_value=0)
mask_under_5 = (pct_safe[cats] < 5).any(axis=1)
users_all_three_but_under_5 = pct_safe.index[have_all_three & mask_under_5].tolist()

print(f"Users missing at least one category ({len(users_missing_categories)}):")
print(users_missing_categories)

print(f"\nUsers with all three categories but one or more < 5% ({len(users_all_three_but_under_5)}):")
print(users_all_three_but_under_5)

# (Optional) show detailed tables for inspection
missing_tbl = pivot_safe.loc[users_missing_categories, cats].astype(int)
under5_tbl  = pct_safe.loc[users_all_three_but_under_5, cats].round(2)

# Uncomment to display nicely in notebooks:
# display(missing_tbl)
# display(under5_tbl)

# (Optional) save to CSV
# missing_tbl.to_csv("users_missing_categories_counts.csv")
# under5_tbl.to_csv("users_all_three_but_under5_pct.csv")


# --- Filter remaining users ---
# Remaining = users not in either list
excluded_users = set(users_missing_categories) | set(users_all_three_but_under_5)
remaining_users = [u for u in pivot_safe.index if u not in excluded_users]

print(f"Remaining users to plot ({len(remaining_users)}):")
print(remaining_users)

# Restrict pivot_pct to remaining users only
pivot_pct_remaining = pivot_pct.loc[remaining_users]

# Plot stacked horizontal bar chart for remaining users
ax = pivot_pct_remaining.plot(
    kind="barh", stacked=True, figsize=(8,10),
    color={"Kühler":"#1f77b4", "Keine Veränderung":"#ff7f0e", "Wärmer":"#2ca02c"}
)
ax.set_xlabel("Percentage of answers (%)")
ax.set_ylabel("User")
ax.set_title("Thermal Preference Distribution (Filtered Users Only)")
ax.legend(title="Preference", bbox_to_anchor=(1.05,1), loc="lower right")
plt.tight_layout()
plt.show()



### Filter and save the remaining users

In [ ]:
# --- Drop users with missing categories OR <5% class ---

excluded_users = set(users_missing_categories) | set(users_all_three_but_under_5)
df = pd.read_csv(OUT_NO_MISSING_ALLENC)
df_cold = pd.read_csv(OUT_COLD)
df_remaining = df[~df["user_id"].isin(excluded_users)].copy()
df_cold_remaining = df_cold[~df_cold["user_id"].isin(excluded_users)].copy()
print(f"Original dataset: {len(df)} rows")
print(f"Remaining dataset: {len(df_remaining)} rows")
print(f"Dropped users: {len(excluded_users)} -> {sorted(excluded_users)}")
df_remaining.to_csv(OUT_NO_MISSING_ALLENC_FILTERED, index=False)
df_cold_remaining.to_csv(OUT_COLD_FILTERED, index=False)

## Creating subsets of the dataset

In [ ]:

df = pd.read_csv(OUT_NO_MISSING_ALLENC)

INDOOR_SENS     = ["t_ubi","rh_ubi"]
WEARABLE_SENS    = ["hr_ubi"]
PHYS_COLS  = ["sex","height_cm","weight_kg"]
FULL_INDOOR_SENS     = ["t_ubi","rh_ubi","CO2"]
FULL_WEARABLE_SENS = ["hr_ubi","activity"]
FULL_PHYS_COLS = ["sex","height_cm","weight_kg","fitness_level"] 
USER_ID_COL  = ["user_id"]            
TARGET_COL   = ["thermal_cozie"]  

indoor_sens    = [c for c in INDOOR_SENS if c in df.columns]
wearable_sens  = [c for c in WEARABLE_SENS if c in df.columns]
phys_cols = [c for c in PHYS_COLS if c in df.columns]
full_indoor_sens    = [c for c in FULL_INDOOR_SENS if c in df.columns]
full_wearable_sens  = [c for c in FULL_WEARABLE_SENS if c in df.columns]
full_phys_cols = [c for c in FULL_PHYS_COLS if c in df.columns]
user_id_col = [c for c in USER_ID_COL if c in df.columns]
target_col  = [c for c in TARGET_COL if c in df.columns]

def build_cols(*groups):
    seen = set()
    out = []
    for grp in groups:
        for c in grp:
            if c not in seen:
                seen.add(c)
                out.append(c)
    return out

subsets = {
    "indoor_sens": build_cols(user_id_col, indoor_sens, target_col),
    "wearable_phys": build_cols(user_id_col, wearable_sens, phys_cols, target_col),
    "phys_cols": build_cols(user_id_col, phys_cols, target_col),
    "full_indoor_sens": build_cols(user_id_col, full_indoor_sens, target_col),
    "full_wearable_sens": build_cols(user_id_col, full_wearable_sens, target_col),
    "full_phys_cols": build_cols(user_id_col, full_phys_cols, target_col),
    "indoor_wearable": build_cols(user_id_col, full_indoor_sens, full_wearable_sens, target_col),
}

os.makedirs(OUT_DIR, exist_ok=True)

for name, cols in subsets.items():
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"Skipping {name} (missing columns: {missing})")
        continue
    out_path = os.path.join(OUT_DIR, f"{name}.csv")
    df[cols].to_csv(out_path, index=False)
    print(f"Saved {name} → {out_path}")


### Dorn: survey file for the DORN experiments
The fixed Big Five scores (`DORN_qns_b5p_calc_fixed.csv`, from `preprocessing_dorn.ipynb`) are merged with `sex` and `swls` from `DORN_qns_hsps_swls_phy.csv` into `DORN_qns_b5p_calc_sex_swls_fixed.csv`. This one file serves all DORN runs; `qns_features` in `dorn_config.yaml` selects the survey columns.

The Big Five + `swls` file (`DORN_qns_b5p_calc_swls_fixed.csv`) is no longer generated; its lines below are commented out.

In [ ]:
df_qns_swls = pd.read_csv(DORN_QNS_SWLS,usecols=["user_id","sex","swls"])
df_qns = pd.read_csv(DORN_QNS)

In [ ]:
df_qns.merge(df_qns_swls,on= "user_id").to_csv(OUT_DORN_ALL, index=False)
# not needed any more: dorn_config.yaml uses the file above and selects columns with qns_features
# df_qns.merge(df_qns_swls.loc[:,["user_id","swls"]],on= "user_id").to_csv(OUT_DORN_SURVEY, index=False)